# Training and preparing the models

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
import torch.optim as optim  # Added missing import
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt  # For visualization
import torchvision

## Define FineTuneResNet50 (using the previously provided resnet code)
This is our fine-tuned model from training.
It wraps a pretrained ResNet50 (in the attribute base_model) and replaces the final fc layer.
Note: The base layers were frozen during initial training.
class FineTuneResNet50(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.2):
        super(FineTuneResNet50, self).__init__()
        self.base_model = models.resnet50(pretrained=True)
        # Freeze base layers (if needed; here we assume weights are already trained)
        for param in self.base_model.parameters():
            param.requires_grad = False
        # Replace final fully connected layer with dropout and linear layer
        in_features = self.base_model.fc.in_features
        self.base_model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, num_classes)
        )
    
    def forward(self, x):
        return self.base_model(x)
    
    def unfreeze_layers(self, num_layers_to_unfreeze=20):
        children = list(self.base_model.children())
        total_layers = len(children)
        for child in children[max(0, total_layers - num_layers_to_unfreeze):]:
            for param in child.parameters():
                param.requires_grad = True

## Functions for Model and Dataset Loading

In [ ]:
def get_model(model_name, num_classes, pretrained=True):
    """
    Returns the specified fine-tuned model.
    For resnet50, we use our FineTuneResNet50.
    """
    model_name = model_name.lower()
    if model_name == 'resnet50':
        model = FineTuneResNet50(num_classes=num_classes, dropout_rate=0.2)
    else:
        raise ValueError("Only 'resnet50' is supported in this configuration.")
    return model

def get_dataset(dataset_name, image_size=224):
    """
    Loads the dataset with separate transforms for training and testing.
    For 'cifar10' we use CIFAR10; for 'imagenet' we assume ImageFolder paths.
    """
    if dataset_name.lower() == 'cifar10':
        # Training transforms with data augmentation
        train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), 
                                 (0.2023, 0.1994, 0.2010)),
        ])

        # Testing transforms without augmentation
        test_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), 
                                 (0.2023, 0.1994, 0.2010)),
        ])

        train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
        test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
        num_classes = 10

    elif dataset_name.lower() == 'imagenet':
        # For ImageNet, use ImageFolder with proper paths (update paths as needed)
        train_transform = transforms.Compose([
            transforms.RandomResizedCrop(image_size),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
        test_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
        train_dataset = datasets.ImageFolder(root='./data/imagenet/train', transform=train_transform)
        test_dataset = datasets.ImageFolder(root='./data/imagenet/val', transform=test_transform)
        num_classes = 1000
    else:
        raise ValueError("Unsupported dataset. Choose from 'cifar10' or 'imagenet'.")

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)
    return train_loader, test_loader, num_classes

def load_trained_model(model_name, dataset_name, num_classes, device):
    """
    Loads the trained model with saved weights.
    We assume weights were saved as "best_{dataset_name}_{model_name}_model.pth".
    """
    model = get_model(model_name, num_classes, pretrained=True).to(device)
    model_path = f"best_{dataset_name}_{model_name}_model.pth"
    print("Loading ", model_path)
    try:
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Loaded model weights from '{model_path}' successfully.")
    except FileNotFoundError:
        print(f"Model weights file '{model_path}' not found.")
        raise
    model.eval()  # Set model to evaluation mode
    return model

## Partitioning the Model into 3 Devices
We partition the fine-tuned ResNet50 (i.e. its base_model) as follows:
  - Device 1: conv1, bn1, relu, maxpool, layer1
  - Device 2: layer2 and layer3
  - Device 3: layer4, AdaptiveAvgPool, Flatten, fc

In [ ]:
class PartitionedResNet50(nn.Module):
    def __init__(self, full_model, device):
        super(PartitionedResNet50, self).__init__()
        self.device = device

        # Device 1: initial layers
        self.device1 = nn.Sequential(
            full_model.base_model.conv1,
            full_model.base_model.bn1,
            full_model.base_model.relu,
            full_model.base_model.maxpool,
            full_model.base_model.layer1
        ).to(device)

        # Device 2: second and third blocks combined
        self.device2 = nn.Sequential(
            full_model.base_model.layer2,
            full_model.base_model.layer3
        ).to(device)

        # Device 3: fourth block and final classification layers
        self.device3 = nn.Sequential(
            full_model.base_model.layer4,
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            full_model.base_model.fc
        ).to(device)

    def forward(self, x):
        x = x.to(self.device)
        x = self.device1(x)
        x = self.device2(x)
        x = self.device3(x)
        return x

## Evaluation Functions

In [ ]:
def evaluate_full_model(full_model, test_loader, device):
    full_model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images = images.to(device)
            labels = labels.to(device)

            outputs = full_model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(test_loader):
                print(f"Full Model: Processed {total} samples")

    accuracy = 100 * correct / total
    print(f"Full Model Test Accuracy: {accuracy:.2f}%")
    return accuracy

def evaluate_partitioned_model(partitioned_model, test_loader, device):
    partitioned_model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images = images.to(device)
            labels = labels.to(device)

            outputs = partitioned_model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(test_loader):
                print(f"Partitioned Model: Processed {total} samples")

    accuracy = 100 * correct / total
    print(f"Partitioned Model Test Accuracy: {accuracy:.2f}%")
    return accuracy

def compare_models(full_model, partitioned_model, test_loader, device, atol=1e-4):
    """
    Compares outputs of the full and partitioned models.
    """
    full_model.eval()
    partitioned_model.eval()

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images = images.to(device)
            # Full model prediction
            outputs_full = full_model(images)
            # Partitioned model prediction
            outputs_partitioned = partitioned_model(images)

            if not torch.allclose(outputs_full, outputs_partitioned, atol=atol):
                max_diff = (outputs_full - outputs_partitioned).abs().max().item()
                print("Outputs differ between full and partitioned models.")
                print(f"Max difference: {max_diff}")
            else:
                print("Outputs match between full and partitioned models.")

            # Compare only on the first batch
            break

## Main Execution Block

In [ ]:
# Configuration
model_name = 'resnet50'  # Using our fine-tuned resnet50 model
# Set dataset_name to 'cifar10' or 'imagenet' as desired
dataset_name = 'cifar10'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
image_size = 224  # For resnet50

# Load dataset
train_loader, test_loader, num_classes = get_dataset(dataset_name, image_size=image_size)

# Load the trained full model (with weights)
full_model = load_trained_model(model_name, dataset_name, num_classes, device)

# Evaluate the full model
full_accuracy = evaluate_full_model(full_model, test_loader, device)

# Create the partitioned model (3-device partition)
partitioned_model = PartitionedResNet50(full_model, device)
partitioned_model.eval()

# Evaluate the partitioned model
partitioned_accuracy = evaluate_partitioned_model(partitioned_model, test_loader, device)

# Compare outputs between full and partitioned models
compare_models(full_model, partitioned_model, test_loader, device)